# QuVINE: Comprehensive Embedding Methods Comparison

This notebook demonstrates and compares multiple embedding methods:
- **Classical Random Walk (RWR)**
- **Discrete-Time Quantum Walk (DTQW)**
- **Continuous-Time Quantum Walk (CTQW)**
- **Node2Vec** (baseline)
- **Fused QuVINE** (combining multiple methods)

We'll test these on random graphs with known structures.

## ⚠️ IMPORTANT: Before Running
**Please restart the kernel and run all cells from the beginning** to ensure the configuration is properly loaded.

In Jupyter: `Kernel → Restart & Run All`

## Setup and Imports

In [5]:
import sys
import os
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from omegaconf import DictConfig

# Add src to path
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

# QuVINE imports
from quvine.data import (
    generate_barabasi_albert,
    generate_modular_network,
    generate_graph_with_seeds_and_targets,
)
from quvine.views.generator import ViewBuilder
from quvine.walks.base import BaseWalker
from quvine.embedding.word2vec import corpus_to_embedding
from quvine.baselines import run_node2vec, run_netmf
from quvine.fusion.fuse import fuse_embeddings
from quvine.evaluation.ranking import (
    seed_centroid_scores,
    max_seed_cosine_scores,
    evaluate_embeddings_ranking
)
from quvine.utils.seed import set_global_seed

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Imports successful")

✓ Imports successful


## Configuration

In [6]:
# Create configuration
cfg = DictConfig({
    "seed": 42,
    
    "walks": {
        "kinds": ["rwr", "ctqw", "dtqw"],
        "num_walks": 5,
        "walk_length": 40,
        "restart_prob": 0.15,
        "max_iter": 1000,
        "steps": 20,
        "time": 1.2,
        "coin": "grover"
    },
    
    "views": {
        "num_views": 3,
        "constrained": True,
        "max_degree": 9,
        "max_nodes": 80,
        "max_edges": 350,
        "degree_norm": True,
        "degree_alpha": 0.7
    },
    
    "train": {
        "embedding_dim": 128,
        "window": 10,
        "sg": 1,
        "negative": 5,
        "min_count": 1,
        "workers": 4,
        "epochs": 10
    },
    
    "baselines": {
        "node2vec": {
            "enabled": True,
            "dimensions": 128,
            "walk_length": 40,
            "num_walks": 10,
            "p": 1.0,
            "q": 1.0,
            "window": 10,
            "min_count": 1,
            "workers": 4,
            "seed": 42
        }
    },
    
    "fusion": {
        "method": "concatenate",
        "weights": None
    }
})

# Set global seed
set_global_seed(cfg.seed)

print("Configuration:")
print(f"  Walk types: {cfg.walks.kinds}")
print(f"  Embedding dim: {cfg.train.embedding_dim}")
print(f"  Node2Vec enabled: {cfg.baselines.node2vec.enabled}")

Configuration:
  Walk types: ['rwr', 'ctqw', 'dtqw']
  Embedding dim: 128
  Node2Vec enabled: True


## Part 1: Generate Random Graphs

In [7]:
# Parameters
n_nodes = 200
num_seeds = 20
num_targets = 30
seed = cfg.seed

print(f"Generating graphs with {n_nodes} nodes")
print(f"Seeds: {num_seeds}, Targets: {num_targets}")
print("="*60)

# Generate Scale-Free network
print("\n1. Scale-Free Network (Barabási-Albert)")
G_ba, seeds_ba, targets_ba = generate_graph_with_seeds_and_targets(
    n=n_nodes,
    num_seeds=num_seeds,
    num_targets=num_targets,
    graph_type='barabasi_albert',
    seed=seed,
    m=3
)
print(f"   Nodes: {G_ba.number_of_nodes()}, Edges: {G_ba.number_of_edges()}")
print(f"   Seeds: {len(seeds_ba)}, Targets: {len(targets_ba)}")

# Generate Modular network
print("\n2. Modular Network")
G_mod, seeds_mod, targets_mod = generate_graph_with_seeds_and_targets(
    n=n_nodes,
    num_seeds=num_seeds,
    num_targets=num_targets,
    graph_type='modular',
    seed=seed,
    num_communities=5,
    nodes_per_community=40,
    p_intra=0.3,
    p_inter=0.01
)
print(f"   Nodes: {G_mod.number_of_nodes()}, Edges: {G_mod.number_of_edges()}")
print(f"   Seeds: {len(seeds_mod)}, Targets: {len(targets_mod)}")

print("\n" + "="*60)
print("✓ Graphs generated successfully")

Generating graphs with 200 nodes
Seeds: 20, Targets: 30

1. Scale-Free Network (Barabási-Albert)
   Nodes: 200, Edges: 591
   Seeds: 20, Targets: 30

2. Modular Network
   Nodes: 200, Edges: 1358
   Seeds: 20, Targets: 30

✓ Graphs generated successfully


## Part 2: Generate Walks with Multiple Methods

In [8]:
def generate_all_walks(G, seeds, cfg, seed=42):
    """
    Generate walks using RWR, DTQW, and CTQW methods.
    
    Parameters
    ----------
    G : nx.Graph
        Input graph
    seeds : list
        Seed nodes to start walks from
    cfg : DictConfig
        Configuration
    seed : int
        Random seed
        
    Returns
    -------
    dict
        Dictionary mapping walk type to list of walks
    """
    rng = np.random.default_rng(seed)
    view_gen = ViewBuilder(cfg=cfg, rng=rng)
    walker = BaseWalker(cfg=cfg, rng=rng)
    
    all_walks = {k: [] for k in cfg.walks.kinds}
    
    print(f"Generating walks for {len(seeds)} seed nodes...")
    
    for i, root in enumerate(seeds):
        if (i + 1) % 5 == 0:
            print(f"  Processed {i + 1}/{len(seeds)} seeds")
        
        # Build views for this seed
        views = view_gen.build(G, root)
        
        for view in views:
            view_g = G.subgraph(view)
            view_nodes = list(view_g.nodes())
            
            if len(view_nodes) < 2 or view_g.number_of_edges() == 0:
                continue
            
            # Run walker on this view
            out = walker.run(G, root, view_nodes)
            
            for walk_kind, walks in out.items():
                all_walks[walk_kind].extend(walks)
    
    return all_walks

# Generate walks for both graphs
print("Scale-Free Network:")
walks_ba = generate_all_walks(G_ba, seeds_ba, cfg, seed=cfg.seed)

print("\nModular Network:")
walks_mod = generate_all_walks(G_mod, seeds_mod, cfg, seed=cfg.seed)

# Display statistics
print("\n" + "="*60)
print("Walk Statistics:")
print(f"{'Walk Type':<15} {'Scale-Free':<15} {'Modular':<15}")
print("-"*45)
for kind in cfg.walks.kinds:
    print(f"{kind.upper():<15} {len(walks_ba[kind]):<15} {len(walks_mod[kind]):<15}")

print("\n✓ Walks generated successfully")

Scale-Free Network:
Generating walks for 20 seed nodes...
  Processed 5/20 seeds
  Processed 10/20 seeds
  Processed 15/20 seeds
  Processed 20/20 seeds

Modular Network:
Generating walks for 20 seed nodes...
  Processed 5/20 seeds
  Processed 10/20 seeds
  Processed 15/20 seeds
  Processed 20/20 seeds

Walk Statistics:
Walk Type       Scale-Free      Modular        
---------------------------------------------
RWR             300             300            
CTQW            300             300            
DTQW            300             300            

✓ Walks generated successfully


## Part 3: Train Embeddings

### 3.1 QuVINE Embeddings (RWR, DTQW, CTQW)

In [13]:
def train_embeddings_from_walks(G, all_walks, cfg):
    """
    Train Word2Vec embeddings from walks.
    
    Parameters
    ----------
    G : nx.Graph
        Input graph
    all_walks : dict
        Dictionary mapping walk type to walks
    cfg : DictConfig
        Configuration
        
    Returns
    -------
    dict
        Dictionary mapping walk type to embedding matrix
    """
    embeddings = {}
    nodes = [str(n) for n in G.nodes()]  # Convert to strings
    
    for kind, walks in all_walks.items():
        if len(walks) == 0:
            print(f"  Warning: No walks for {kind}, skipping")
            continue
        
        print(f"  Training {kind.upper()} embedding from {len(walks)} walks...")
        
        # Convert walks to strings
        walks_str = [[str(node) for node in walk] for walk in walks]
        
        Z = corpus_to_embedding(
            corpus=walks_str,
            nodes=nodes,
            vector_size=cfg.train.embedding_dim,
            window=cfg.train.window,
            sg=cfg.train.sg,
            negative=cfg.train.negative,
            min_count=cfg.train.min_count,
            workers=cfg.train.workers,
            epochs=cfg.train.epochs
        )
        
        embeddings[kind] = Z
        print(f"    Shape: {Z.shape}")
    
    return embeddings

print("Training QuVINE embeddings...\n")

print("Scale-Free Network:")
embeddings_ba = train_embeddings_from_walks(G_ba, walks_ba, cfg)

print("\nModular Network:")
embeddings_mod = train_embeddings_from_walks(G_mod, walks_mod, cfg)

print("\n✓ QuVINE embeddings trained")

Training QuVINE embeddings...

Scale-Free Network:
  Training RWR embedding from 300 walks...


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


    Shape: (200, 128)
  Training CTQW embedding from 300 walks...
    Shape: (200, 128)
  Training DTQW embedding from 300 walks...
    Shape: (200, 128)

Modular Network:
  Training RWR embedding from 300 walks...
    Shape: (200, 128)
  Training CTQW embedding from 300 walks...
    Shape: (200, 128)
  Training DTQW embedding from 300 walks...
    Shape: (200, 128)

✓ QuVINE embeddings trained


### 3.2 Node2Vec Baseline

In [14]:
if cfg.baselines.node2vec.enabled:
    print("Training Node2Vec embeddings...\n")
    
    print("Scale-Free Network:")
    Z_n2v_ba = run_node2vec(
        graph=G_ba,
        nodes=list(G_ba.nodes()),
        dimensions=cfg.baselines.node2vec.dimensions,
        walk_length=cfg.baselines.node2vec.walk_length,
        num_walks=cfg.baselines.node2vec.num_walks,
        p=cfg.baselines.node2vec.p,
        q=cfg.baselines.node2vec.q,
        window=cfg.baselines.node2vec.window,
        min_count=cfg.baselines.node2vec.min_count,
        workers=cfg.baselines.node2vec.workers,
        seed=cfg.baselines.node2vec.seed
    )
    embeddings_ba['node2vec'] = Z_n2v_ba
    print(f"  Shape: {Z_n2v_ba.shape}")
    
    print("\nModular Network:")
    Z_n2v_mod = run_node2vec(
        graph=G_mod,
        nodes=list(G_mod.nodes()),
        dimensions=cfg.baselines.node2vec.dimensions,
        walk_length=cfg.baselines.node2vec.walk_length,
        num_walks=cfg.baselines.node2vec.num_walks,
        p=cfg.baselines.node2vec.p,
        q=cfg.baselines.node2vec.q,
        window=cfg.baselines.node2vec.window,
        min_count=cfg.baselines.node2vec.min_count,
        workers=cfg.baselines.node2vec.workers,
        seed=cfg.baselines.node2vec.seed
    )
    embeddings_mod['node2vec'] = Z_n2v_mod
    print(f"  Shape: {Z_n2v_mod.shape}")
    
    print("\n✓ Node2Vec embeddings trained")
else:
    print("Node2Vec disabled in configuration")

Training Node2Vec embeddings...

Scale-Free Network:


Computing transition probabilities:   0%|          | 0/200 [00:00<?, ?it/s]

Generating walks (CPU: 4): 100%|██████████| 2/2 [00:00<00:00, 43.14it/s]
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


  Shape: (200, 128)

Modular Network:


Computing transition probabilities:   0%|          | 0/200 [00:00<?, ?it/s]

Generating walks (CPU: 4): 100%|██████████| 2/2 [00:00<00:00, 64.17it/s]
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


  Shape: (200, 128)

✓ Node2Vec embeddings trained


### 3.3 NetMF Baseline

In [ ]:
print("Training NetMF embeddings...\n")

print("Scale-Free Network:")
Z_netmf_ba = run_netmf(
    graph=G_ba,
    nodes=list(G_ba.nodes()),
    dimensions=128,
    window_size=10,
    negative=1,
    seed=cfg.seed
)
embeddings_ba['netmf'] = Z_netmf_ba
print(f"  Shape: {Z_netmf_ba.shape}")

print("\nModular Network:")
Z_netmf_mod = run_netmf(
    graph=G_mod,
    nodes=list(G_mod.nodes()),
    dimensions=128,
    window_size=10,
    negative=1,
    seed=cfg.seed
)
embeddings_mod['netmf'] = Z_netmf_mod
print(f"  Shape: {Z_netmf_mod.shape}")

print("\n✓ NetMF embeddings trained")
print("\nNote: NetMF uses matrix factorization of the DeepWalk matrix,")
print("providing a closed-form solution without random walks.")

### 3.4 Fused Embeddings

In [15]:
from quvine.fusion.fuse import fuse_embeddings_svd

print("Creating fused embeddings...\n")

print("Scale-Free Network:")
# Get list of embeddings (excluding node2vec if present)
emb_list_ba = [embeddings_ba[k] for k in ['rwr', 'ctqw', 'dtqw'] if k in embeddings_ba]
if cfg.fusion.method == 'concatenate':
    fused_ba = np.concatenate(emb_list_ba, axis=1)
elif cfg.fusion.method == 'average':
    fused_ba = np.mean(emb_list_ba, axis=0)
elif cfg.fusion.method == 'svd':
    k = min(Z.shape[1] for Z in emb_list_ba)
    fused_ba = fuse_embeddings_svd(emb_list_ba, k)
else:
    # Default to concatenate
    fused_ba = np.concatenate(emb_list_ba, axis=1)
embeddings_ba['fused'] = fused_ba
print(f"  Fused {len(emb_list_ba)} embeddings using {cfg.fusion.method}")
print(f"  Shape: {fused_ba.shape}")

print("\nModular Network:")
emb_list_mod = [embeddings_mod[k] for k in ['rwr', 'ctqw', 'dtqw'] if k in embeddings_mod]
if cfg.fusion.method == 'concatenate':
    fused_mod = np.concatenate(emb_list_mod, axis=1)
elif cfg.fusion.method == 'average':
    fused_mod = np.mean(emb_list_mod, axis=0)
elif cfg.fusion.method == 'svd':
    k = min(Z.shape[1] for Z in emb_list_mod)
    fused_mod = fuse_embeddings_svd(emb_list_mod, k)
else:
    fused_mod = np.concatenate(emb_list_mod, axis=1)
embeddings_mod['fused'] = fused_mod
print(f"  Fused {len(emb_list_mod)} embeddings using {cfg.fusion.method}")
print(f"  Shape: {fused_mod.shape}")

print("\n✓ Fused embeddings created")

Creating fused embeddings...

Scale-Free Network:
  Fused 3 embeddings using concatenate
  Shape: (200, 384)

Modular Network:
  Fused 3 embeddings using concatenate
  Shape: (200, 384)

✓ Fused embeddings created


## Part 4: Evaluate Embeddings

### 4.1 Gene/Node Prioritization Task

In [16]:
def evaluate_all_embeddings(embeddings, seeds, targets, k_values=[10, 20, 50]):
    """
    Evaluate all embeddings on gene prioritization task.
    
    Parameters
    ----------
    embeddings : dict
        Dictionary mapping method name to embedding matrix
    seeds : list
        Seed nodes
    targets : list
        Target nodes (ground truth)
    k_values : list
        Top-k values to evaluate
        
    Returns
    -------
    pd.DataFrame
        Results dataframe
    """
    results = []
    targets_set = set(targets)
    
    for method_name, Z in embeddings.items():
        print(f"  Evaluating {method_name}...")
        
        # Compute scores using max seed cosine similarity
        scores = max_seed_cosine_scores(Z, seeds)
        
        # Get ranked nodes (excluding seeds)
        n_nodes = len(scores)
        all_nodes = list(range(n_nodes))
        seeds_set = set(seeds)
        candidates = [i for i in all_nodes if i not in seeds_set]
        
        # Rank candidates by score
        ranked = sorted(candidates, key=lambda i: scores[i], reverse=True)
        
        # Evaluate at different k values
        for k in k_values:
            top_k = set(ranked[:k])
            hits = top_k & targets_set
            
            precision = len(hits) / k if k > 0 else 0
            recall = len(hits) / len(targets_set) if len(targets_set) > 0 else 0
            f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
            
            results.append({
                'method': method_name,
                'k': k,
                'precision': precision,
                'recall': recall,
                'f1': f1,
                'num_targets_found': len(hits)
            })
    
    return pd.DataFrame(results)

print("Evaluating embeddings on gene prioritization task...\n")

print("Scale-Free Network:")
results_ba = evaluate_all_embeddings(embeddings_ba, seeds_ba, targets_ba)

print("\nModular Network:")
results_mod = evaluate_all_embeddings(embeddings_mod, seeds_mod, targets_mod)

print("\n✓ Evaluation complete")

Evaluating embeddings on gene prioritization task...

Scale-Free Network:
  Evaluating rwr...


TypeError: evaluate_embeddings_ranking() got an unexpected keyword argument 'scores'

### 4.2 Results Comparison

In [ ]:
# Display results for k=20
k_display = 20

print(f"\nResults at k={k_display}:")
print("\nScale-Free Network:")
display_ba = results_ba[results_ba['k'] == k_display].sort_values('f1', ascending=False)
print(display_ba[['method', 'precision', 'recall', 'f1', 'num_targets_found']].to_string(index=False))

print("\nModular Network:")
display_mod = results_mod[results_mod['k'] == k_display].sort_values('f1', ascending=False)
print(display_mod[['method', 'precision', 'recall', 'f1', 'num_targets_found']].to_string(index=False))

### 4.3 Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Scale-Free results
for method in results_ba['method'].unique():
    data = results_ba[results_ba['method'] == method]
    axes[0].plot(data['k'], data['f1'], marker='o', label=method)

axes[0].set_xlabel('k (Top-k predictions)')
axes[0].set_ylabel('F1 Score')
axes[0].set_title('Scale-Free Network')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Modular results
for method in results_mod['method'].unique():
    data = results_mod[results_mod['method'] == method]
    axes[1].plot(data['k'], data['f1'], marker='o', label=method)

axes[1].set_xlabel('k (Top-k predictions)')
axes[1].set_ylabel('F1 Score')
axes[1].set_title('Modular Network')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✓ Visualization complete")

## Part 5: Graph Complexity Analysis

### 5.1 Compute Complexity Metrics

In [ ]:
from quvine.data.graph_complexity import compute_graph_complexity_metrics
from quvine.data.qbc_complexity import compute_qbc_complexity_from_laplacian

print("Computing graph complexity metrics...\n")

print("Scale-Free Network:")
# Compute standard complexity metrics
complexity_ba = compute_graph_complexity_metrics(G_ba)
print("  Computed spectral and centrality metrics")

# Add QBioCode complexity
try:
    qbc_ba = compute_qbc_complexity_from_laplacian(G_ba, normalized=True, mean_center=True)
    complexity_ba.update(qbc_ba)
    print("  Added QBioCode complexity metrics")
except Exception as e:
    print(f"  QBioCode computation skipped: {e}")

print("\nModular Network:")
# Compute standard complexity metrics
complexity_mod = compute_graph_complexity_metrics(G_mod)
print("  Computed spectral and centrality metrics")

# Add QBioCode complexity
try:
    qbc_mod = compute_qbc_complexity_from_laplacian(G_mod, normalized=True, mean_center=True)
    complexity_mod.update(qbc_mod)
    print("  Added QBioCode complexity metrics")
except Exception as e:
    print(f"  QBioCode computation skipped: {e}")

print("\n✓ Complexity analysis complete")

### 5.2 Complexity Comparison

In [ ]:
# Create comparison dataframe
complexity_comparison = pd.DataFrame({
    'Metric': list(complexity_ba.keys()),
    'Scale-Free': list(complexity_ba.values()),
    'Modular': list(complexity_mod.values())
})

# Calculate difference only for numeric values
def safe_subtract(a, b):
    try:
        return float(a) - float(b)
    except (ValueError, TypeError):
        return 'N/A'

complexity_comparison['Difference'] = [
    safe_subtract(sf, mod) 
    for sf, mod in zip(complexity_comparison['Scale-Free'], complexity_comparison['Modular'])
]

print("\nGraph Complexity Comparison:")
print("="*80)
print(complexity_comparison.to_string(index=False))
print("="*80)

# Highlight key differences
print("\nKey Structural Differences:")
spectral_gap_diff = complexity_ba['spectral_gap'] - complexity_mod['spectral_gap']
print(f"  Spectral Gap: Scale-Free is {abs(spectral_gap_diff):.4f} {'higher' if spectral_gap_diff > 0 else 'lower'}")

alg_conn_diff = complexity_ba['algebraic_connectivity'] - complexity_mod['algebraic_connectivity']
print(f"  Algebraic Connectivity: Scale-Free is {abs(alg_conn_diff):.4f} {'higher' if alg_conn_diff > 0 else 'lower'}")

if 'Isomap Reconstruction Error' in complexity_ba:
    qbc_diff = complexity_ba['Isomap Reconstruction Error'] - complexity_mod['Isomap Reconstruction Error']
    print(f"  QBioCode Complexity: Scale-Free is {abs(qbc_diff):.4f} {'higher' if qbc_diff > 0 else 'lower'}")

### 5.3 Complexity vs Performance Correlation

In [ ]:
# Get best F1 scores for each graph
k_analysis = 20
best_f1_ba = results_ba[results_ba['k'] == k_analysis].groupby('method')['f1'].max()
best_f1_mod = results_mod[results_mod['k'] == k_analysis].groupby('method')['f1'].max()

print(f"\nPerformance Analysis (k={k_analysis}):")
print("="*60)

print("\nScale-Free Network (Higher Spectral Gap):")
print(f"  Spectral Gap: {complexity_ba['spectral_gap']:.4f}")
print(f"  Best F1 Score: {best_f1_ba.max():.4f} ({best_f1_ba.idxmax()})")
print(f"  Average F1: {best_f1_ba.mean():.4f}")

print("\nModular Network (More Community Structure):")
print(f"  Spectral Gap: {complexity_mod['spectral_gap']:.4f}")
print(f"  Best F1 Score: {best_f1_mod.max():.4f} ({best_f1_mod.idxmax()})")
print(f"  Average F1: {best_f1_mod.mean():.4f}")

print("\nInsights:")
if best_f1_ba.mean() > best_f1_mod.mean():
    print("  → Embeddings perform better on scale-free networks")
    print("  → Higher spectral gap may facilitate better node representations")
else:
    print("  → Embeddings perform better on modular networks")
    print("  → Community structure may help with node prioritization")

# Method-specific analysis
print("\nMethod Performance Across Graph Types:")
for method in best_f1_ba.index:
    if method in best_f1_mod.index:
        diff = best_f1_ba[method] - best_f1_mod[method]
        better_on = 'Scale-Free' if diff > 0 else 'Modular'
        print(f"  {method:12s}: Better on {better_on:12s} (Δ={abs(diff):.4f})")

### 5.4 Complexity Visualization

In [ ]:
# Get all numeric metrics (exclude strings like 'num_nodes', 'num_edges')
all_metrics = []
for key in complexity_ba.keys():
    try:
        # Try to convert to float - if it works, it's numeric
        float(complexity_ba[key])
        float(complexity_mod[key])
        # Exclude basic counts
        if key not in ['num_nodes', 'num_edges']:
            all_metrics.append(key)
    except (ValueError, TypeError):
        pass

if len(all_metrics) > 0:
    # Create multiple subplots if there are many metrics
    n_metrics = len(all_metrics)
    
    if n_metrics <= 10:
        # Single plot for up to 10 metrics
        fig, ax = plt.subplots(figsize=(max(12, n_metrics * 1.2), 6))
        
        x = np.arange(n_metrics)
        width = 0.35
        
        values_ba = [complexity_ba[m] for m in all_metrics]
        values_mod = [complexity_mod[m] for m in all_metrics]
        
        ax.bar(x - width/2, values_ba, width, label='Scale-Free', alpha=0.8)
        ax.bar(x + width/2, values_mod, width, label='Modular', alpha=0.8)
        
        ax.set_xlabel('Complexity Metric', fontsize=12)
        ax.set_ylabel('Value', fontsize=12)
        ax.set_title('Complete Graph Complexity Comparison', fontsize=14, fontweight='bold')
        ax.set_xticks(x)
        ax.set_xticklabels([m.replace('_', ' ').title() for m in all_metrics], 
                          rotation=45, ha='right', fontsize=10)
        ax.legend(fontsize=11)
        ax.grid(True, alpha=0.3, axis='y')
        
        plt.tight_layout()
        plt.show()
    else:
        # Multiple plots for many metrics (split into groups)
        metrics_per_plot = 10
        n_plots = (n_metrics + metrics_per_plot - 1) // metrics_per_plot
        
        fig, axes = plt.subplots(n_plots, 1, figsize=(14, 5 * n_plots))
        if n_plots == 1:
            axes = [axes]
        
        for plot_idx in range(n_plots):
            start_idx = plot_idx * metrics_per_plot
            end_idx = min(start_idx + metrics_per_plot, n_metrics)
            plot_metrics = all_metrics[start_idx:end_idx]
            
            x = np.arange(len(plot_metrics))
            width = 0.35
            
            values_ba = [complexity_ba[m] for m in plot_metrics]
            values_mod = [complexity_mod[m] for m in plot_metrics]
            
            axes[plot_idx].bar(x - width/2, values_ba, width, label='Scale-Free', alpha=0.8)
            axes[plot_idx].bar(x + width/2, values_mod, width, label='Modular', alpha=0.8)
            
            axes[plot_idx].set_xlabel('Complexity Metric', fontsize=11)
            axes[plot_idx].set_ylabel('Value', fontsize=11)
            axes[plot_idx].set_title(f'Complexity Comparison (Part {plot_idx + 1}/{n_plots})', 
                                    fontsize=12, fontweight='bold')
            axes[plot_idx].set_xticks(x)
            axes[plot_idx].set_xticklabels([m.replace('_', ' ').title() for m in plot_metrics], 
                                          rotation=45, ha='right', fontsize=9)
            axes[plot_idx].legend(fontsize=10)
            axes[plot_idx].grid(True, alpha=0.3, axis='y')
        
        plt.tight_layout()
        plt.show()
    
    print(f"\n✓ Visualized all {n_metrics} complexity metrics")
else:
    print("No numeric complexity metrics available for visualization")

## Part 6: Summary and Insights

In [ ]:
print("="*60)
print("SUMMARY")
print("="*60)

print("\nEmbedding Methods Tested:")
for i, method in enumerate(embeddings_ba.keys(), 1):
    print(f"  {i}. {method.upper()}")

print("\nBest Performing Methods (F1 @ k=20):")
print("\nScale-Free Network:")
best_ba = display_ba.iloc[0]
print(f"  {best_ba['method']}: F1={best_ba['f1']:.4f}, Precision={best_ba['precision']:.4f}, Recall={best_ba['recall']:.4f}")

print("\nModular Network:")
best_mod = display_mod.iloc[0]
print(f"  {best_mod['method']}: F1={best_mod['f1']:.4f}, Precision={best_mod['precision']:.4f}, Recall={best_mod['recall']:.4f}")

print("\nKey Insights:")
print("  - Quantum walks (DTQW, CTQW) capture different structural properties than classical RWR")
print("  - Fused embeddings often outperform individual methods")
print("  - Performance varies by graph structure (scale-free vs modular)")
print("  - Node2Vec provides a strong baseline for comparison")

print("\n" + "="*60)
print("✓ Analysis complete!")
print("="*60)